In [1]:
import pandas as pd
import numpy as np

In [2]:
models = [
    "linear_model",
    "nam",
    "nbm",
    "ebm",
    "aplr",
    "pymgcv",
    "paramb _order0_cont-1",
    "paramb _order1_cont-1",
    "paramb _order1_cont0",
    "paramb _order2_cont-1",
    "paramb _order2_cont0",
    "paramb _order2_cont1",
    "paramb _order3_cont-1",
    "paramb _order3_cont0",
    "paramb _order3_cont1",
    "paramb _order3_cont2",
] 
datasets = [
    "housing",
    "concrete",
    "power",
    "energy",
    "protein",
    "msd",
    "fraud",
    "diabetes",
    "cover",
    "har",
    "lpmc",
]
name_data = {
    "housing": "Housing",
    "concrete": "Concrete",
    "power": "Power",
    "energy": "Energy",
    "protein": "Protein",
    "msd": "Year",
    "fraud": "Fraud",
    "diabetes": "Diabetes",
    "cover": "Cover",
    "har": "HAR",
    "lpmc": "LPMC",
}
name_model = {
    "linear_model": "Linear/Logistic regression",
    "nam": "NAM",
    "nbm": "NBM",
    "ebm": "EBM",
    "aplr": "APLR",
    "pymgcv": "PyMGCV",
    "paramb _order0_cont-1": "ParamB-$O^0C^{-1}$",
    "paramb _order1_cont-1": "ParamB-$O^1C^{-1}$",
    "paramb _order1_cont0": "ParamB-$O^1C^0$",
    "paramb _order2_cont-1": "ParamB-$O^2C^{-1}$",
    "paramb _order2_cont0": "ParamB-$O^2C^0$",
    "paramb _order2_cont1": "ParamB-$O^2C^1$",
    "paramb _order3_cont-1": "ParamB-$O^3C^{-1}$",
    "paramb _order3_cont0": "ParamB-$O^3C^0$",
    "paramb _order3_cont1": "ParamB-$O^3C^1$",
    "paramb _order3_cont2": "ParamB-$O^3C^2$",
}  

paramb_models = [
    "ParamB-$O^0C^{-1}$",
    "ParamB-$O^1C^{-1}$",
    "ParamB-$O^1C^0$",
    "ParamB-$O^2C^{-1}$",
    "ParamB-$O^2C^0$",
    "ParamB-$O^2C^1$",
    "ParamB-$O^3C^{-1}$",
    "ParamB-$O^3C^0$",
    "ParamB-$O^3C^1$",
    "ParamB-$O^3C^2$",
]

trials = 10

results = pd.DataFrame(columns=["train_score", "val_score", "test_score", "time"])
results_iter = pd.DataFrame(columns=["best_iter"])
for d in datasets:
    for i in range(trials):
        min_val_score = np.inf
        for m in models:
            if i > 0 and d in ["protein", "msd", "fraud", "diabetes", "cover", "har", "lpmc"]:
                continue  # only one trial for large datasets
            if "paramb-test" in m:
                m, str_m = m.split(" ")
                cols = ["train_score", "val_score", "test_score", "time", "best_iter"]
            elif "paramb" in m:
                m, str_m = m.split(" ")
                cols = ["train_score", "val_score", "test_score", "time", "best_iter"]
            else:
                m, str_m = m, ""
                cols = ["train_score", "val_score", "test_score", "time"]
            path = f"{d}/{i}/{m}/results{str_m}.csv"
            print(f"Loading {path}...")
            try:
                df = pd.read_csv(path)
                if m == "paramb":
                    if df["val_score"].iloc[0] < min_val_score:
                        df_min = df.copy()
                        min_val_score = df["val_score"].iloc[0]
                if "paramb" in m:
                    m = f"{m} {str_m}"
                    df_iter = df[["best_iter"]]
                    df_iter = df_iter.set_index(
                        pd.MultiIndex.from_tuples(
                            [(name_data[d], name_model[m], i)],
                            names=["dataset", "model", "trial"],
                        )
                    )
                    results_iter = pd.concat([results_iter, df_iter])

                df = df[cols].set_index(
                    pd.MultiIndex.from_tuples(
                        [(name_data[d], name_model[m], i)],
                        names=["dataset", "model", "trial"],
                    )
                )
                results = pd.concat([results, df])
                if m == "paramb _order3_cont2":
                    df = df_min[cols].set_index(
                        pd.MultiIndex.from_tuples(
                            [(name_data[d], "ParamB", i)],
                            names=["dataset", "model", "trial"],
                        )
                    )
                    results = pd.concat([results, df])
            except FileNotFoundError:
                df = pd.DataFrame(
                    np.nan,
                    index=pd.MultiIndex.from_tuples(
                        [(name_data[d], name_model[m], i)],
                        names=["dataset", "model", "trial"],
                    ),
                    columns=["train_score", "val_score", "test_score", "time"],
                )
                results = pd.concat([results, df])
                if "paramb" in m:
                    m = f"{m} {str_m}"
                    df_iter = pd.DataFrame(
                        np.nan,
                        index=pd.MultiIndex.from_tuples(
                            [(name_data[d], name_model[m], i)],
                            names=["dataset", "model", "trial"],
                        ),
                        columns=["best_iter"],
                    )
                    results_iter = pd.concat([results_iter, df_iter])
results

Loading housing/0/linear_model/results.csv...
Loading housing/0/nam/results.csv...
Loading housing/0/nbm/results.csv...
Loading housing/0/ebm/results.csv...
Loading housing/0/aplr/results.csv...
Loading housing/0/pymgcv/results.csv...
Loading housing/0/paramb/results_order0_cont-1.csv...
Loading housing/0/paramb/results_order1_cont-1.csv...
Loading housing/0/paramb/results_order1_cont0.csv...
Loading housing/0/paramb/results_order2_cont-1.csv...
Loading housing/0/paramb/results_order2_cont0.csv...
Loading housing/0/paramb/results_order2_cont1.csv...
Loading housing/0/paramb/results_order3_cont-1.csv...
Loading housing/0/paramb/results_order3_cont0.csv...
Loading housing/0/paramb/results_order3_cont1.csv...
Loading housing/0/paramb/results_order3_cont2.csv...
Loading housing/1/linear_model/results.csv...
Loading housing/1/nam/results.csv...
Loading housing/1/nbm/results.csv...
Loading housing/1/ebm/results.csv...
Loading housing/1/aplr/results.csv...
Loading housing/1/pymgcv/results.csv

/tmp/ipykernel_104364/360230546.py:121: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, df])


Loading housing/4/nam/results.csv...
Loading housing/4/nbm/results.csv...
Loading housing/4/ebm/results.csv...
Loading housing/4/aplr/results.csv...
Loading housing/4/pymgcv/results.csv...
Loading housing/4/paramb/results_order0_cont-1.csv...
Loading housing/4/paramb/results_order1_cont-1.csv...
Loading housing/4/paramb/results_order1_cont0.csv...
Loading housing/4/paramb/results_order2_cont-1.csv...
Loading housing/4/paramb/results_order2_cont0.csv...
Loading housing/4/paramb/results_order2_cont1.csv...
Loading housing/4/paramb/results_order3_cont-1.csv...
Loading housing/4/paramb/results_order3_cont0.csv...
Loading housing/4/paramb/results_order3_cont1.csv...
Loading housing/4/paramb/results_order3_cont2.csv...
Loading housing/5/linear_model/results.csv...
Loading housing/5/nam/results.csv...
Loading housing/5/nbm/results.csv...
Loading housing/5/ebm/results.csv...
Loading housing/5/aplr/results.csv...
Loading housing/5/pymgcv/results.csv...
Loading housing/5/paramb/results_order0_co

,train_score,val_score,test_score,time,best_iter
"(Housing, Linear/Logistic regression, 0)",20.815317,27.103025,23.371266,0.001349,NaN
"(Housing, NAM, 0)",13.115855,13.189297,18.378486,0.508575,NaN
"(Housing, NBM, 0)",13.827425,13.582468,19.094835,0.649641,NaN
"(Housing, EBM, 0)",8.452616,11.505924,14.454111,0.149483,NaN
"(Housing, APLR, 0)",12.012526,10.836776,15.828011,0.784677,NaN
...,...,...,...,...,...
"(LPMC, ParamB-$O^3C^{-1}$, 0)",0.636382,0.649771,0.673591,30.636343,1241.0
"(LPMC, ParamB-$O^3C^0$, 0)",0.643016,0.646805,0.671366,47.032385,6150.0
"(LPMC, ParamB-$O^3C^1$, 0)",0.645429,0.645148,0.672188,47.486104,11906.0
"(LPMC, ParamB-$O^3C^2$, 0)",0.648901,0.648159,0.673594,47.424667,18136.0


In [3]:
new_index = pd.MultiIndex.from_tuples(
    results.index, names=["dataset", "model", "trial"]
)

results = results.set_index(new_index)

In [4]:
group_object = results.reset_index(level="dataset")[["dataset", "test_score"]].pivot(columns="dataset", values="test_score").groupby("model")

formatted_df = group_object.mean().map(lambda x: f"{x:.3f}" if not pd.isna(x) else "-") + " (± " + group_object.std().map(lambda x: f"{x:.3f}" if not pd.isna(x) else "-") + ")"

In [5]:
formatted_df_cols_inplace = formatted_df.loc[
    [
        "Linear/Logistic regression",
        "EBM",
        "NAM",
        "NBM",
        "APLR",
        "PyMGCV",
        "ParamB-$O^3C^{-1}$",
    ],
    [
        "Housing",
        "Concrete",
        "Power",
        "Energy",
        "Protein",
        "Year",
        "Fraud",
        "Diabetes",
        "Cover",
        "HAR",
        "LPMC",
    ],
].copy()

In [6]:
formatted_df_cols_inplace = formatted_df_cols_inplace.swapaxes(0, 1)

/tmp/ipykernel_104364/7773540.py:1: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  formatted_df_cols_inplace = formatted_df_cols_inplace.swapaxes(0, 1)


In [7]:
pd.set_option("display.max_columns", None)
formatted_df_cols_inplace

model,Linear/Logistic regression,EBM,NAM,NBM,APLR,PyMGCV,ParamB-$O^3C^{-1}$
dataset,,,,,,,
Housing,22.016 (± 6.538),13.691 (± 4.323),19.052 (± 4.215),21.013 (± 9.798),14.087 (± 3.113),21.982 (± 6.407),13.683 (± 4.108)
Concrete,112.939 (± 9.648),28.153 (± 3.366),52.213 (± 5.415),48.165 (± 6.278),34.009 (± 3.616),113.030 (± 9.512),27.774 (± 3.594)
Power,21.029 (± 0.791),13.386 (± 0.651),20.757 (± 1.240),19.263 (± 0.575),17.770 (± 0.678),21.030 (± 0.786),15.859 (± 0.686)
Energy,9.103 (± 1.442),1.229 (± 0.176),8.562 (± 1.435),10.662 (± 3.340),1.110 (± 0.196),9.001 (± 1.338),1.215 (± 0.143)
Protein,27.193 (± -),23.494 (± -),24.224 (± -),30.901 (± -),24.718 (± -),27.186 (± -),23.584 (± -)
Year,90.442 (± -),85.274 (± -),86.343 (± -),94.412 (± -),85.488 (± -),90.426 (± -),85.395 (± -)
Fraud,0.009 (± -),0.003 (± -),0.003 (± -),0.012 (± -),0.003 (± -),0.004 (± -),0.003 (± -)
Diabetes,0.416 (± -),0.379 (± -),0.387 (± -),0.385 (± -),0.387 (± -),0.416 (± -),0.380 (± -)
Cover,0.951 (± -),0.599 (± -),0.613 (± -),0.618 (± -),- (± -),0.000 (± -),0.579 (± -)


In [8]:
formatted_df_cols_inplace.index.name = ""
formatted_df_cols_inplace.columns.name = ""
formatted_df_cols_inplace.to_latex("results.tex", escape=False, multicolumn_format="c", multirow=True, column_format="l" + "c" * 10)

In [9]:
group_object_time = (
    results.reset_index(level="dataset")[["dataset", "time"]]
    .pivot(columns="dataset", values="time")
    .groupby("model")
)

formatted_df_time = (
    group_object_time.mean().map(lambda x: f"{x:.3f}" if not pd.isna(x) else "-")
    + " (± "
    + group_object_time.std().map(lambda x: f"{x:.3f}" if not pd.isna(x) else "-")
    + ")"
)
formatted_df_time_inplace = formatted_df_time.loc[
    [
        "Linear/Logistic regression",
        "EBM",
        "NAM",
        "NBM",
        "APLR",
        "PyMGCV",
        "ParamB-$O^3C^{-1}$",
    ],
    [
        "Housing",
        "Concrete",
        "Power",
        "Energy",
        "Protein",
        "Year",
        "Fraud",
        "Diabetes",
        "Cover",
        "HAR",
        "LPMC",
    ],
].copy()

In [10]:
formatted_df_time_inplace = formatted_df_time_inplace.swapaxes(0, 1)

/tmp/ipykernel_104364/3721951330.py:1: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  formatted_df_time_inplace = formatted_df_time_inplace.swapaxes(0, 1)


In [11]:
formatted_df_time_inplace.index.name = ""
formatted_df_time_inplace.columns.name = ""
formatted_df_time_inplace.to_latex("results_time.tex", escape=False, multicolumn_format="c", multirow=True, column_format="l" + "c" * 10)

In [12]:
formatted_df_time_inplace

,Linear/Logistic regression,EBM,NAM,NBM,APLR,PyMGCV,ParamB-$O^3C^{-1}$
,,,,,,,
Housing,0.001 (± 0.000),0.271 (± 0.255),0.352 (± 0.147),0.453 (± 0.171),0.978 (± 0.856),0.016 (± 0.001),2.338 (± 3.562)
Concrete,0.001 (± 0.000),1.163 (± 1.207),0.414 (± 0.145),0.507 (± 0.125),7.125 (± 2.788),0.017 (± 0.001),3.368 (± 3.293)
Power,0.001 (± 0.000),5.156 (± 0.921),0.915 (± 0.351),1.355 (± 0.643),19.485 (± 7.085),0.053 (± 0.014),6.492 (± 4.575)
Energy,0.002 (± 0.001),1.005 (± 1.094),0.399 (± 0.133),0.407 (± 0.130),0.979 (± 0.134),0.013 (± 0.001),1.556 (± 2.611)
Protein,0.004 (± -),2.298 (± -),9.292 (± -),5.740 (± -),91.163 (± -),0.742 (± -),25.829 (± -)
Year,0.308 (± -),66.343 (± -),230.872 (± -),255.686 (± -),2133.798 (± -),121.807 (± -),611.758 (± -)
Fraud,1.117 (± -),10.429 (± -),67.630 (± -),15.004 (± -),1005.472 (± -),10.027 (± -),22.359 (± -)
Diabetes,1.064 (± -),7.660 (± -),82.196 (± -),53.178 (± -),310.639 (± -),40.006 (± -),207.932 (± -)
Cover,8.728 (± -),371.311 (± -),329.540 (± -),135.860 (± -),- (± -),0.000 (± -),1078.351 (± -)


In [13]:
group_object = results.reset_index(level="dataset")[["dataset", "val_score"]].pivot(columns="dataset", values="val_score").groupby("model")

formatted_df = group_object.mean().map(lambda x: f"{x:.3f}" if not pd.isna(x) else "-") + " (± " + group_object.std().map(lambda x: f"{x:.3f}" if not pd.isna(x) else "-") + ")"

In [14]:
formatted_df_cols_inplace = formatted_df.loc[
    [
        "Linear/Logistic regression",
        "EBM",
        "NAM",
        "NBM",
        "APLR",
        "PyMGCV",
        "ParamB-$O^3C^{-1}$",
    ],
    [
        "Housing",
        "Concrete",
        "Power",
        "Energy",
        "Protein",
        "Year",
        "Fraud",
        "Diabetes",
        "Cover",
        "HAR",
        "LPMC",
    ],
].copy()

In [15]:
formatted_df_cols_inplace = formatted_df_cols_inplace.swapaxes(0, 1)

/tmp/ipykernel_104364/7773540.py:1: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  formatted_df_cols_inplace = formatted_df_cols_inplace.swapaxes(0, 1)


In [16]:
pd.set_option("display.max_columns", None)
formatted_df_cols_inplace

model,Linear/Logistic regression,EBM,NAM,NBM,APLR,PyMGCV,ParamB-$O^3C^{-1}$
dataset,,,,,,,
Housing,20.898 (± 6.441),10.473 (± 2.997),14.713 (± 6.082),17.981 (± 6.104),11.479 (± 4.314),23.006 (± 6.831),12.045 (± 4.391)
Concrete,103.985 (± 14.585),24.797 (± 6.127),50.568 (± 10.201),44.914 (± 9.001),35.597 (± 6.398),106.965 (± 14.900),27.846 (± 8.078)
Power,20.574 (± 1.117),12.785 (± 1.216),20.097 (± 1.635),18.730 (± 1.159),17.286 (± 1.128),20.594 (± 1.119),15.391 (± 1.323)
Energy,8.073 (± 1.632),1.069 (± 0.355),7.664 (± 1.345),9.055 (± 3.219),0.925 (± 0.310),8.287 (± 1.595),1.064 (± 0.359)
Protein,26.598 (± -),22.752 (± -),23.785 (± -),30.197 (± -),24.133 (± -),26.615 (± -),22.800 (± -)
Year,92.963 (± -),86.647 (± -),88.220 (± -),96.449 (± -),86.814 (± -),93.020 (± -),86.697 (± -)
Fraud,0.008 (± -),0.002 (± -),0.002 (± -),0.012 (± -),0.003 (± -),0.004 (± -),0.003 (± -)
Diabetes,0.413 (± -),0.382 (± -),0.387 (± -),0.389 (± -),0.389 (± -),0.415 (± -),0.382 (± -)
Cover,0.953 (± -),0.599 (± -),0.613 (± -),0.617 (± -),- (± -),inf (± -),0.579 (± -)


In [17]:
formatted_df_cols_inplace.index.name = ""
formatted_df_cols_inplace.columns.name = ""
formatted_df_cols_inplace.to_latex("results_val.tex", escape=False, multicolumn_format="c", multirow=True, column_format="l" + "c" * 10)

In [18]:
group_object = results.reset_index(level="dataset")[["dataset", "train_score"]].pivot(columns="dataset", values="train_score").groupby("model")

formatted_df = group_object.mean().map(lambda x: f"{x:.3f}" if not pd.isna(x) else "-") + " (± " + group_object.std().map(lambda x: f"{x:.3f}" if not pd.isna(x) else "-") + ")"

In [19]:
formatted_df_cols_inplace = formatted_df.loc[
    [
        "Linear/Logistic regression",
        "EBM",
        "NAM",
        "NBM",
        "APLR",
        "PyMGCV",
        "ParamB-$O^3C^{-1}$",
    ],
    [
        "Housing",
        "Concrete",
        "Power",
        "Energy",
        "Protein",
        "Year",
        "Fraud",
        "Diabetes",
        "Cover",
        "HAR",
        "LPMC",
    ],
].copy()

In [20]:
formatted_df_cols_inplace = formatted_df_cols_inplace.swapaxes(0, 1)

/tmp/ipykernel_104364/7773540.py:1: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  formatted_df_cols_inplace = formatted_df_cols_inplace.swapaxes(0, 1)


In [21]:
pd.set_option("display.max_columns", None)
formatted_df_cols_inplace

model,Linear/Logistic regression,EBM,NAM,NBM,APLR,PyMGCV,ParamB-$O^3C^{-1}$
dataset,,,,,,,
Housing,22.344 (± 1.590),8.700 (± 3.873),17.816 (± 2.659),19.080 (± 4.191),11.575 (± 1.438),22.149 (± 1.483),8.310 (± 2.674)
Concrete,106.516 (± 2.827),14.506 (± 1.806),48.906 (± 3.300),44.671 (± 6.431),29.757 (± 2.297),106.319 (± 2.802),17.237 (± 4.026)
Power,20.725 (± 0.236),10.814 (± 0.851),20.101 (± 0.669),18.769 (± 0.413),17.094 (± 0.260),20.724 (± 0.236),14.251 (± 0.483)
Energy,8.578 (± 0.429),0.993 (± 0.040),8.379 (± 0.799),10.156 (± 2.495),0.910 (± 0.045),8.473 (± 0.377),0.997 (± 0.034)
Protein,26.817 (± -),22.353 (± -),23.903 (± -),30.362 (± -),24.338 (± -),26.815 (± -),22.529 (± -)
Year,91.013 (± -),83.620 (± -),85.666 (± -),93.787 (± -),84.447 (± -),91.009 (± -),83.462 (± -)
Fraud,0.009 (± -),0.002 (± -),0.002 (± -),0.013 (± -),0.003 (± -),0.004 (± -),0.003 (± -)
Diabetes,0.414 (± -),0.371 (± -),0.382 (± -),0.379 (± -),0.384 (± -),0.414 (± -),0.371 (± -)
Cover,0.955 (± -),0.595 (± -),0.611 (± -),0.617 (± -),- (± -),0.000 (± -),0.572 (± -)


In [22]:
formatted_df_cols_inplace.index.name = ""
formatted_df_cols_inplace.columns.name = ""
formatted_df_cols_inplace.to_latex("results_train.tex", escape=False, multicolumn_format="c", multirow=True, column_format="l" + "c" * 10)